In [1]:
import jax
import netket as nk

import numpy as np
import jax.numpy as jnp

# from neuralimportancesampling._src.driver.ngd_antoine.grad_sample.models import PCMolecule
from neuralimportancesampling._src.driver.ngd_antoine.grad_sample.models import PCMolecule 

In [2]:
mol, mo_coeff, mf = PCMolecule.molecule(cid=62714)#62714
molecule = PCMolecule(mol=mol, mo_coeff=mo_coeff)

H = molecule.hamiltonian.to_jax_operator()
hi = molecule.hilbert_space

using 2d
Hartree-Fock energy: -7.767362135748573
E(CCSD) = -7.784454825913955  E_corr = -0.01709269016538233
CCSD energy: -7.784454825913955


/Users/lucagravina/venvs/neuralimportancesampling/lib/python3.12/site-packages/jax/_src/ops/scatter.py:108: FutureWarning: scatter inputs have incompatible types: cannot safely cast value from dtype=int64 to dtype=bool with jax_numpy_dtype_promotion='standard'. In future JAX releases this will result in an error.
  warnings.warn(


In [3]:
all_states = hi.all_states()
print("all_states_size =", all_states.shape)

print("n_fermions =", hi.n_fermions)
print("n_orbitals =", hi.n_orbitals)
print("n_connected_states =", H.max_conn_size)

n_max = hi.n_orbitals * 2

all_states_size = (225, 12)
n_fermions = 4
n_orbitals = 6
n_connected_states = 115


In [4]:
_operator_data = H._operator_data
print("_operator_data keys =", _operator_data.keys())

_operator_data keys = dict_keys(['diag', 'offdiag'])


In [5]:
_operator_data['offdiag'].keys()

dict_keys([2, 4])

In [6]:
from jax import Array
from functools import partial
@jax.jit
def hamming_distance(x, y):
    """
    Args:
        x, y: 1D JAX arrays of 0/1 values with the same shape.

    Returns:
      d : Hamming distance (scalar)
    """
    return jnp.sum(x != y)
  
@partial(jax.jit, static_argnames=['k'])
def select_changes(x: Array, y: Array, k: int = 4):
    diff = x - y  
    size = k // 2
    k_destroy = jnp.argwhere(diff == 1, size=size, fill_value=-1).reshape(-1)
    l_create = jnp.argwhere(diff == -1, size=size, fill_value=-1).reshape(-1)
    return jnp.flip(k_destroy), jnp.flip(l_create)

In [7]:
x = all_states[-1]
xp, mels = H.get_conn_padded(x)

print(x,"\n")

y = xp[75]
print(y,"\n")

is_y_in_xp = jnp.any(jnp.all(xp == y, axis=1))
print("Is y in xp?", is_y_in_xp)

if is_y_in_xp:
    y_index = jnp.where(jnp.all(xp == y, axis=1))[0]
    y_index = y_index[0]
    print("matrix element connecting x to y =", mels[y_index])


print("\nHamming distance =", hamming_distance(x, y))

[1 1 0 0 0 0 1 1 0 0 0 0] 

[1 0 0 1 0 0 0 1 0 1 0 0] 

Is y in xp? True
matrix element connecting x to y = -0.008311547637045058

Hamming distance = 4


In [8]:
k_destroy, l_create = select_changes(x, y)
print("x = ", x)
print("y = ", y)
print("Deletions indices =", k_destroy)
print("Creations indices =", l_create)

x =  [1 1 0 0 0 0 1 1 0 0 0 0]
y =  [1 0 0 1 0 0 0 1 0 1 0 0]
Deletions indices = [6 1]
Creations indices = [9 3]


In [9]:
index_array, create_array, weight_array = _operator_data['offdiag'][4]
index_array.shape

(12, 12)

In [10]:
ind = index_array[tuple(k_destroy)]
weight_array[ind]

Array([ 0.16278426, -0.00194991, -0.08876831, -0.03169328, -0.01485794,
        0.00654162, -0.01254777, -0.01365813,  0.02570631, -0.00062031,
        0.01296156, -0.00831155, -0.00831155, -0.01098745, -0.00542389,
        0.00411286, -0.00841146], dtype=float64)

In [11]:
creates = create_array[ind]  # shape (n_max, 2)
creates

Array([[ 6,  0],
       [ 6,  2],
       [ 6,  5],
       [ 7,  0],
       [ 7,  1],
       [ 7,  2],
       [ 7,  5],
       [ 8,  0],
       [ 8,  1],
       [ 8,  2],
       [ 8,  5],
       [ 9,  3],
       [10,  4],
       [11,  0],
       [11,  1],
       [11,  2],
       [11,  5]], dtype=int64)

In [12]:
base = n_max # or just use 12
creates_1d = creates[:, 0] * base + creates[:, 1]
target_1d = l_create[0] * base + l_create[1]  # 9*12 + 3 = 111

idx = jnp.searchsorted(creates_1d, target_1d)

In [13]:
print("weight_array shape =", weight_array.shape)
weight_array[ind,idx]

weight_array shape = (65, 17)


Array(-0.00831155, dtype=float64)

In [14]:
@jax.jit
def jw_sign_fast(x, k_destroy, l_create):
    """Vectorized JW sign computation."""
    # Cumsum gives number of particles to the left of each site
    cumsum = jnp.cumsum(x)
    
    # Parity from destruction (use cumsum - x to exclude site itself)
    left_counts = jnp.concatenate([jnp.array([0]), cumsum[:-1]])
    parity_destroy = jnp.sum(left_counts[k_destroy])
    
    # After destruction
    xd = x.at[k_destroy].set(0)
    cumsum_d = jnp.cumsum(xd)
    left_counts_d = jnp.concatenate([jnp.array([0]), cumsum_d[:-1]])
    parity_create = jnp.sum(left_counts_d[l_create])
    
    return 1 - 2 * ((parity_destroy + parity_create) % 2)


jw_sign_fast(x, k_destroy, l_create)

Array(1, dtype=int64)

In [15]:
from netket.jax import COOArray
from netket.utils.types import Array


@partial(jax.jit, static_argnums=0)
@partial(jnp.vectorize, signature="(n)->()", excluded=(0, 1, 3, 4, 5))
def _get_mel_offdiagonal(
    k: int,
    x: Array,
    y: Array,
    index_array: Array | COOArray | None,
    create_array: Array | None,
    weight_array: Array,
) -> Array:
    r"""
    Get the matrix element between two states `x` and `y` for operators that
    change `k` particles, i.e. operators of the form
    
    .. math::
        c^\dagger_{i_1} ... c^\dagger_{i_{k/2}} c_{j_1} ... c_{j_{k/2}}
        
    If the Hamming distance between `x` and `y` is not `k`, returns 0.0.
    
    Args:
        k: int
            Number of particles changed by the operator (must be even).
        x: Array
            Initial state (1D array of 0/1 values).
        y: Array
            Final state (1D array of 0/1 values).
        index_array: Array | COOArray | None
            Precomputed index array for the operator for the selected order 'k'.
        create_array: Array | None
            Precomputed creation array for the operator for the selected order 'k'.
        weight_array: Array
            Precomputed weight array for the operator for the selected order 'k'.
            
    Returns:
        mel: Array
            The matrix element connecting `x` to `y`.
    """
    
    k_destroy, l_create = select_changes(x, y, k=k)
    ind = index_array[tuple(k_destroy)]
    
    creates = create_array[ind] # shape (n_max, k)
    idx = jnp.argmax((creates == l_create).all(axis=1))
    
    sgn = jw_sign_fast(x, k_destroy, l_create)
    result = sgn * weight_array[ind, idx]
    return result
    # return jax.lax.select(
    #     hamming_distance(x, y) == k,
    #     result,
    #     jnp.zeros_like(result)
    # )

In [16]:
k = 4
_get_mel_offdiagonal(
    k,
    x,
    y,
    index_array,
    create_array,
    weight_array,
)

Array(-0.00831155, dtype=float64)

In [17]:
def filter_keys(pytree, filter_func:callable):
    result = {}
    for outer_key, inner_dict in pytree.items():
        result[outer_key] = {
            k: v for k, v in inner_dict.items() 
            if filter_func(k)
        }
    return result


_operator_data_filtered = filter_keys(_operator_data, lambda k: k == 4)
_operator_data_filtered['diag'] = {}

In [18]:
from netket.experimental.operator._particle_number_conserving_fermionic._kernels import get_conn_padded_pnc

x = jnp.array([1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0])
y = jnp.array([1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0])

xp, mels = get_conn_padded_pnc(_operator_data_filtered, x, hi.n_fermions)
is_y_in_xp = jnp.any(jnp.all(xp == y, axis=1))
print("Is y in xp?", is_y_in_xp)

if is_y_in_xp:
    y_index = jnp.where(jnp.all(xp == y, axis=1))
    y_index = y_index
    print("matrix element connecting x to y =", mels[y_index])


print("\nHamming distance =", hamming_distance(x, y))
print("Even though the Hamming distance is 2, the operator changes 4 particles. The only difference is that now one particle is destroyed and created in the same site.")
print("This excitation still belongs to the 4-particle change off-diagonal terms.")

Is y in xp? True
matrix element connecting x to y = [0.07778086 0.08876831 0.15993537]

Hamming distance = 2
Even though the Hamming distance is 2, the operator changes 4 particles. The only difference is that now one particle is destroyed and created in the same site.
This excitation still belongs to the 4-particle change off-diagonal terms.


In [19]:
index_array, create_array, weight_array = _operator_data_filtered['offdiag'][4]


In [20]:
k_destroy_, l_create_ = select_changes(x, y, k=2)
print("x = ", x)
print("y = ", y)
print("Deletions indices =", k_destroy_)
print("Creations indices =", l_create_)

x =  [1 1 0 0 0 0 1 1 0 0 0 0]
y =  [1 0 0 0 0 1 1 1 0 0 0 0]
Deletions indices = [1]
Creations indices = [5]


In [21]:
possible_same_particle_transitions = jnp.argwhere(x & y)
print("Possible same-particle transitions (indices) =\n", possible_same_particle_transitions)

Possible same-particle transitions (indices) =
 [[0]
 [6]
 [7]]


In [22]:
result = []
for idx in possible_same_particle_transitions.flatten():
    k_destroy = jnp.sort(jnp.array([k_destroy_[0], idx]), descending=True)
    l_create = jnp.sort(jnp.array([l_create_[0], idx]), descending=True)
    print("destroy", k_destroy)
    print("create ", l_create)
    print("---")
    
    ind = index_array[tuple(k_destroy)]
    creates = create_array[ind] # shape (n_max, k)
    
    # serch sorted does not work here because of the [0,0] padding
    # base = n_max
    # creates_1d = creates[:, 0] * base + creates[:, 1]
    # target_1d = l_create[0] * base + l_create[1] # flattening preserves original lexicographic order
    # idx = jnp.searchsorted(creates_1d, target_1d, )
    
    idx = jnp.argmax((creates == l_create).all(axis=1))
    
    sgn = jw_sign_fast(x, k_destroy, l_create)
    result.append(sgn * weight_array[ind, idx])

destroy [1 0]
create  [5 0]
---
destroy [6 1]
create  [6 5]
---
destroy [7 1]
create  [7 5]
---


In [23]:
result

[Array(0.07778086, dtype=float64),
 Array(0.08876831, dtype=float64),
 Array(0.15993537, dtype=float64)]

In [24]:
sum(result)

Array(0.32648454, dtype=float64)

In [25]:
k_destroy_, l_create_ = select_changes(x, y, k=2)
k_destroy_.shape

(1,)

In [26]:
same_sites = jnp.argwhere(x & y, size=3)
same_sites.shape

(3, 1)

In [27]:
k_destroy = jnp.sort(jnp.hstack([same_sites, jnp.full((same_sites.shape[0], 1), k_destroy_[0])]), axis=1, descending=True)
l_create  = jnp.sort(jnp.hstack([same_sites, jnp.full((same_sites.shape[0], 1), l_create_[0])]), axis=1, descending=True)
k_destroy.shape

(3, 2)

In [28]:
ind = index_array[k_destroy[:, 0], k_destroy[:, 1]]
ind.shape

(3,)

In [29]:
creates = create_array[ind]
creates.shape

(3, 17, 2)

In [30]:
jnp.argmax(jnp.all(creates == l_create[..., None, :], axis=-1), axis=-1)

Array([2, 2, 6], dtype=int64)

In [31]:
mask = jnp.all(creates == l_create[..., None, :], axis=-1)
jnp.sum(jnp.arange(creates.shape[-2]) * mask, axis=-1)

Array([2, 2, 6], dtype=int64)

In [32]:
jw_sign_fast(x, k_destroy, l_create)

Array(1, dtype=int64)

In [33]:
l_create[0].shape

(2,)

In [34]:
weight_array[ind, idx]

Array([ 0.        , -0.01254777, -0.15993537], dtype=float64)

In [35]:
from netket.experimental.operator._particle_number_conserving_fermionic._kernels import get_conn_padded_pnc

x = jnp.array([1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0])
y = x

xp, mels = get_conn_padded_pnc(_operator_data_filtered, x, hi.n_fermions)
is_y_in_xp = jnp.any(jnp.all(xp == y, axis=1))
print("Is y in xp?", is_y_in_xp)

if is_y_in_xp:
    y_index = jnp.where(jnp.all(xp == y, axis=1))
    y_index = y_index
    print("matrix element connecting x to y =", mels[y_index])


print("\nHamming distance =", hamming_distance(x, y))

Is y in xp? True
matrix element connecting x to y = [-0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0. -0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.
  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.  0.]

Hamming distance = 0


In [36]:
occupied = jnp.where(x)[0]
print("occupied =", occupied)

i, j = jnp.triu_indices(len(occupied), k=1)
destroy_create = jnp.sort(jnp.stack([occupied[i], occupied[j]], axis=1), axis=1, descending=True, )
print("destroy_create =\n", destroy_create)

occupied = [0 1 6 7]
destroy_create =
 [[1 0]
 [6 0]
 [7 0]
 [6 1]
 [7 1]
 [7 6]]


In [37]:
ind = index_array[k_destroy[:, 0], k_destroy[:, 1]]
print("ind =", ind)
creates = create_array[ind]
print("creates_shape =", creates.shape)

idx = jnp.argmax(jnp.all(creates == l_create[..., None, :], axis=-1), axis=-1)
print("idx =", idx)

mel = jnp.sum(weight_array[ind, idx])

ind = [ 1 16 22]
creates_shape = (3, 17, 2)
idx = [2 2 6]


In [43]:
import jax
import jax.numpy as jnp
from functools import partial

from netket.jax import COOArray
from netket.utils.types import Array

@jax.jit
def hamming_distance(x, y):
    """
    Args:
        x, y: 1D JAX arrays of 0/1 values with the same shape.

    Returns:
      d : Hamming distance (scalar)
    """
    return jnp.sum(x != y)
  
@partial(jax.jit, static_argnames=['k'])
def select_changes(x: Array, y: Array, k: int = 4):
    diff = x - y  
    size = k // 2
    k_destroy = jnp.argwhere(diff == 1, size=size, fill_value=-1).reshape(-1)
    l_create = jnp.argwhere(diff == -1, size=size, fill_value=-1).reshape(-1)
    return jnp.flip(k_destroy), jnp.flip(l_create)


@jax.jit
def jw_sign_fast(x, k_destroy, l_create):
    """
    Fast and correct Jordan–Wigner sign that matches the original implementation.
    """

    prefix = jnp.cumsum(x) - x # number of ones to the left
    parity_destroy = jnp.sum(prefix[k_destroy])

    xd = x.at[k_destroy].set(0) # apply destruction BEFORE computing create parity
    prefix_d = jnp.cumsum(xd) - xd # prefix after destruction

    parity_create = jnp.sum(prefix_d[l_create])

    total_parity = (parity_destroy + parity_create) & 1 # total parity
    return 1 - 2 * total_parity # (-1)^parity


@jax.jit
@partial(jnp.vectorize, signature="(n)->()", excluded=(0, 2, 3, 4))
def _get_mel_twobody(
    x: Array,
    y: Array,
    index_array: Array | COOArray | None,
    create_array: Array | None,
    weight_array: Array,
):
    r"""
    Get the matrix element between two states `x` and `y` for two-body operators
    of the form 
    
    .. math::
        c^\dagger_{i} c^\dagger_{j} c_{k} c_{l}
        
    Args:
        x: Array
            Initial state (1D array of 0/1 values).
        y: Array
            Final state (1D array of 0/1 values).
        index_array: Array | COOArray | None
            Precomputed index array for the two-body operator.
        create_array: Array | None
            Precomputed creation array for the two-body operator.
        weight_array: Array
            Precomputed weight array for the two-body operator.
            
    Returns:
        mel: Array
            The matrix element connecting `x` to `y`.
    """
    def compute(k_destroy, l_create, ind):
        creates = create_array[ind] # shape (n_max, 4)
        
        # idx = jnp.all(creates == l_create[..., None, :], axis=-1)
        mask = jnp.all(creates == l_create[..., None, :], axis=-1)
        idx = jnp.sum(jnp.arange(creates.shape[-2]) * mask, axis=-1)
        
        sgn = jw_sign_fast(x, k_destroy, l_create)
        return sgn * weight_array[ind, idx]

    def case_k4():
        r"""
        Handles the case where the Hamming distance between `x` and `y` is 4,
        corresponding to two-body operator transitions between different sites, i.e
        
        .. math::
            c^\dagger_{i} c^\dagger_{j} c_{k} c_{l} with i,j,k,l all different.
            
        An example of such a transition is:
        x = [1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0]
        y = [1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0]
        """
        k_destroy, l_create = select_changes(x, y, k=4)
        ind = index_array[tuple(k_destroy)]
        return compute(k_destroy, l_create, ind)

    def case_k2():
        r"""
        Handles the case where the Hamming distance between `x` and `y` is 2,
        corresponding to two-body operator transitions that involve the destruction
        and creation of a particle in the same site, i.e
        
        .. math::
            c^\dagger_{i} c^\dagger_{j} c_{j} c_{k}  or  c^\dagger_{i} c^\dagger_{i} c_{k} c_{l}
            
        An example of such a transition is:
        x = [1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0]
        y = [1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0]
        """
        k_destroy_, l_create_ = select_changes(x, y, k=2) # identifies the two sites that are different
        same_sites = jnp.where((x & y), size=3)[0] # all sites where a particle could have been both destroyed and created

        def f_one_site(i):
            r"""
            Computes the matrix element contribution for a single same-site transition.
            Args:
                i: Index of the site where a particle is both destroyed and created.
            Returns:
                Matrix element contribution for this specific same-site transition.
            """
            k_destroy = jnp.sort(jnp.array([k_destroy_[0], i]), descending=True)
            l_create = jnp.sort(jnp.array([l_create_[0], i]), descending=True)
            ind = index_array[tuple(k_destroy)]
            return compute(k_destroy, l_create, ind)

        return jnp.sum(jax.vmap(f_one_site)(same_sites)) # vectorize over all i in same_sites
    
    
    # def case_k0():
    #     r"""
    #     Handles the case where the Hamming distance between `x` and `y` is 0,
    #     corresponding to diagonal matrix elements of two-body operators, i.e
        
    #     .. math::
    #         c^\dagger_{i} c^\dagger_{i} c_{i} c_{i}
    #     """
    #     occupied = jnp.where(x, size=4)[0]
    #     i, j = jnp.triu_indices(len(occupied), k=1)
    #     destroy_create = jnp.sort(jnp.stack([occupied[i], occupied[j]], axis=1), axis=1, descending=True, )

    #     ind = index_array[destroy_create[:, 0], destroy_create[:, 1]]
    #     creates = create_array[ind]
    #     idx = jnp.argmax(jnp.all(creates == destroy_create[..., None, :], axis=-1), axis=-1)

    #     return jnp.sum(weight_array[ind, idx])

    d = hamming_distance(x, y)
    return jnp.where(d == 4, case_k4(), jnp.where(d == 2, case_k2(), 0.0))

    

In [44]:
x = jnp.array([1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0])
# y = jnp.array([1, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0])
y = jnp.array([1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0])


xp, mels = get_conn_padded_pnc(_operator_data_filtered, x, hi.n_fermions)
is_y_in_xp = jnp.any(jnp.all(xp == y, axis=1))
print("Is y in xp?", is_y_in_xp)

if is_y_in_xp:
    y_index = jnp.where(jnp.all(xp == y, axis=1))
    y_index = y_index
    print("matrix element connecting x to y =", mels[y_index], "sum = ", jnp.sum(mels[y_index]))

print("\nHamming distance =", hamming_distance(x, y))

index_array, create_array, weight_array = _operator_data_filtered['offdiag'][4]

_get_mel_twobody(x, y, index_array, create_array, weight_array,)

Is y in xp? True
matrix element connecting x to y = [0.07778086 0.08876831 0.15993537] sum =  0.3264845401504623

Hamming distance = 2


Array(0.32648454, dtype=float64)

In [45]:
def filter_keys(pytree, filter_func:callable):
    result = {}
    for outer_key, inner_dict in pytree.items():
        result[outer_key] = {
            k: v for k, v in inner_dict.items() 
            if filter_func(k)
        }
    return result


_operator_data_filtered = filter_keys(_operator_data, lambda k: k == 4)
_operator_data_filtered['diag'] = {}

In [46]:
import jax.ops
from netket.experimental.operator._particle_number_conserving_fermionic._kernels import get_conn_padded_pnc

x = jnp.array([1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0])

xp, mels = get_conn_padded_pnc(_operator_data_filtered, x, hi.n_fermions)
xp, inverse_indices = jnp.unique(xp, axis=0, return_inverse=True)
mels = jax.ops.segment_sum(mels, inverse_indices, num_segments=len(xp))

print("xp shape =", xp.shape)
print("mels shape =", mels.shape)

xp shape = (35, 12)
mels shape = (35,)


In [47]:
k = 4

index_array, create_array, weight_array = _operator_data_filtered['offdiag'][4]

mels_off_diag = _get_mel_twobody(
    x,
    xp,
    index_array,
    create_array,
    weight_array,
)

np.testing.assert_allclose(mels, mels_off_diag)